# Prompting Techniques: Extracting a Professional Profile

This notebook uses one fictional professional profile to demonstrate how different prompting techniques change a model's response. The recurring task is to extract the person's **work experience** and **technology stack**.

The techniques are patterns for writing instructions; they are not separate model features. We will keep the profile and OpenAI client constant while changing the prompt so the effect is easier to see.

**Setup:** install `openai` and `python-dotenv`, then make `OPENAI_API_KEY` available in your environment or local `.env` file. Each model-call cell sends an API request and may incur charges.

## 1. OpenAI client setup

We use the official OpenAI Python client and Responses API. The helper accepts a prompt and optional instructions, then returns the response text.

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

MODEL = "gpt-4o-mini"

def ask(prompt, *, instructions=None):
    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=prompt,
    )
    return response.output_text

print(f"Client ready. Model: {MODEL}")

python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 33
python-dotenv could not parse statement starting at line 34


Client ready. Model: gpt-4o-mini


## 2. Input: professional profile

This fictional profile includes career history, responsibilities, projects, and technologies. Read it first, then use it as the source for each prompt. The goal is to extract what is stated and avoid inventing details.

In [3]:
profile = """
Professional Profile: Alex Morgan

Summary
Alex is a software engineer with about seven years of experience building web
applications and data services. Alex enjoys working across the stack, improving
reliability, and helping product teams turn customer needs into practical features.

Current Work
Senior Software Engineer at Northstar Health, March 2022–present
- Builds and maintains patient-facing web features and internal operational tools.
- Develops backend APIs in Python using FastAPI and Django.
- Works with PostgreSQL for transactional data and Redis for caching and job queues.
- Deploys services to AWS using Docker and Kubernetes.
- Uses GitHub Actions for continuous integration and Terraform to manage cloud
  infrastructure.
- Collaborates with product managers, designers, QA, and security engineers.
- Reviews code and mentors two junior engineers.
- Recent project: helped split a large Django application into smaller services.
  The migration reduced average API response time by about 30%, according to the
  team's monitoring.

Previous Work
Software Engineer at BrightMarket, July 2019–February 2022
- Built customer and merchant features for an online marketplace.
- Used TypeScript and React for web interfaces, and Node.js with Express for APIs.
- Integrated REST endpoints with third-party payment and shipping providers.
- Wrote automated tests with Jest and Pytest.
- Used PostgreSQL, Redis, Docker, and AWS.
- Worked in two-week Agile sprints and participated in planning and retrospectives.

Earlier Experience
Junior Web Developer at Cedar Labs, June 2017–June 2019
- Created responsive websites using JavaScript, HTML, CSS, and Bootstrap.
- Maintained a small PHP application and MySQL database.
- Used Git for version control and participated in client demonstrations.
- Helped troubleshoot production issues and documented common fixes.

Education and Learning
- Bachelor of Science in Computer Science, completed in 2017.
- Completed online courses in system design and practical machine learning.
- Has experimented with pandas and scikit-learn in personal projects, but has not
  used machine-learning tools in a professional role.

Additional Notes
- Alex is based in Chicago, Illinois.
- Alex has not stated experience with Azure, Go, or mobile app development.
"""

## 3. Zero-shot prompting

A zero-shot prompt states the task but gives no examples. This is the simplest baseline: the model must infer the desired organization from the instruction itself.

In [4]:
zero_shot_prompt = f"""Extract Alex's work experience and technology stack from the profile.
Organize the work experience by role, employer, and dates. Group technologies
into useful categories. Include only information stated in the profile.

Profile:
{profile}"""

print(ask(zero_shot_prompt))

### Alex Morgan's Work Experience

#### Current Work
- **Role:** Senior Software Engineer  
  **Employer:** Northstar Health  
  **Dates:** March 2022 – Present  
  **Responsibilities:**
  - Builds and maintains patient-facing web features and internal operational tools.
  - Develops backend APIs in Python using FastAPI and Django.
  - Works with PostgreSQL for transactional data and Redis for caching and job queues.
  - Deploys services to AWS using Docker and Kubernetes.
  - Uses GitHub Actions for continuous integration and Terraform to manage cloud infrastructure.
  - Collaborates with product managers, designers, QA, and security engineers.
  - Reviews code and mentors two junior engineers.
  - Recent project: Helped split a large Django application into smaller services, reducing average API response time by about 30%.

#### Previous Work
- **Role:** Software Engineer  
  **Employer:** BrightMarket  
  **Dates:** July 2019 – February 2022  
  **Responsibilities:**
  - Built custo

## 4. Few-shot prompting

A few-shot prompt demonstrates the task with examples before the new input. The examples teach the model the output shape and the level of detail to include. They should be representative and correct; poor examples can teach the wrong pattern.

In [5]:
few_shot_prompt = f"""Extract the work experience and technology stack.
Use the headings "Work Experience" and "Technology Stack". Group stack items
by category. Do not add technologies that are not mentioned.

Example 1
Profile:
Mina is a data analyst at Lakeview Retail (2021–2024). She builds reports with
SQL and Excel, uses Python and pandas to clean sales data, and presents findings
to business teams.

Output:
Work Experience:
- Data Analyst, Lakeview Retail (2021–2024): built reports, cleaned sales data,
  and presented findings.

Technology Stack:
- Data and analysis: SQL, Excel, Python, pandas

Example 2
Profile:
Omar worked as a frontend developer at Pinecone Studio from 2020 to 2023.
He built interfaces with Vue.js and JavaScript, styled pages with CSS, and used
GitLab for version control.

Output:
Work Experience:
- Frontend Developer, Pinecone Studio (2020–2023): built and styled web interfaces.

Technology Stack:
- Frontend: Vue.js, JavaScript, CSS
- Version control: GitLab

Now process this profile:
{profile}

Output:"""

print(ask(few_shot_prompt))

**Work Experience:**
- Senior Software Engineer, Northstar Health (March 2022–present): builds and maintains patient-facing web features and internal operational tools, develops backend APIs, collaborates with cross-functional teams, reviews code, and mentors junior engineers.
- Software Engineer, BrightMarket (July 2019–February 2022): built customer and merchant features for an online marketplace, integrated third-party services, wrote automated tests, and worked in Agile sprints.
- Junior Web Developer, Cedar Labs (June 2017–June 2019): created responsive websites, maintained a small PHP application, used version control, and documented common fixes.

**Technology Stack:**
- Backend: Python (FastAPI, Django), Node.js, Express
- Frontend: TypeScript, React, JavaScript, HTML, CSS, Bootstrap
- Databases: PostgreSQL, MySQL
- Caching and job queues: Redis
- Containers and orchestration: Docker, Kubernetes
- Cloud: AWS
- Version control: Git, GitHub Actions
- Infrastructure as code: Terra

## 5. Role or persona prompting

A role prompt gives the model a perspective or focus. Here, the role emphasizes careful résumé-style extraction. A persona can guide tone and attention, but it does not add facts or guarantee accuracy.

In [6]:
role_prompt = f"""Extract Alex's work experience and technology stack.
Use the headings "Work Experience" and "Technology Stack". Keep the answer
concise and include only supported details.

Profile:
{profile}"""

print(ask(
    role_prompt,
    instructions="You are a careful technical recruiter summarizing a candidate profile."
))

**Work Experience**  
- **Senior Software Engineer at Northstar Health** (March 2022–present)  
  - Developed patient-facing web features and backend APIs using Python (FastAPI, Django).  
  - Utilized PostgreSQL, Redis, AWS, Docker, and Kubernetes.  
  - Implemented continuous integration with GitHub Actions and managed infrastructure with Terraform.  
  - Led migration to microservices architecture, reducing API response time by 30%.  

- **Software Engineer at BrightMarket** (July 2019–February 2022)  
  - Built features for an online marketplace using TypeScript, React, Node.js (Express), PostgreSQL, and Redis.  
  - Integrated third-party payment and shipping APIs and automated testing with Jest and Pytest.  

- **Junior Web Developer at Cedar Labs** (June 2017–June 2019)  
  - Developed responsive websites with JavaScript, HTML, CSS, and Bootstrap; maintained a PHP application with MySQL.  

**Technology Stack**  
- **Languages:** Python, TypeScript, JavaScript, HTML, CSS, PHP  


## 6. Instruction and constraint prompting

This technique makes requirements explicit: what to extract, how to format it, and what not to infer. Constraints are especially helpful when the answer must follow a predictable format or avoid unsupported claims.

In [7]:
constraint_prompt = f"""From the profile, extract:
1. Work experience: each role, employer, dates, and a short description.
2. Technology stack: group technologies by category.

Rules:
- Use only facts stated in the profile.
- Do not treat a personal experiment as professional experience.
- Mention when a technology is described only as personal-project experience.
- If a category has no stated tools, omit it.
- Use concise bullet points.

Profile:
{profile}"""

print(ask(constraint_prompt))

### 1. Work Experience

- **Senior Software Engineer**  
  **Employer:** Northstar Health  
  **Dates:** March 2022–present  
  **Description:**  
  - Builds and maintains patient-facing web features and internal operational tools.
  - Develops backend APIs in Python using FastAPI and Django.
  - Works with PostgreSQL for transactional data and Redis for caching and job queues.
  - Deploys services to AWS using Docker and Kubernetes.
  - Uses GitHub Actions for CI and Terraform for cloud infrastructure management.
  - Collaborates with product teams; mentors junior engineers.

- **Software Engineer**  
  **Employer:** BrightMarket  
  **Dates:** July 2019–February 2022  
  **Description:**  
  - Built customer and merchant features for an online marketplace.
  - Utilized TypeScript and React for web interfaces; Node.js with Express for APIs.
  - Integrated REST endpoints with third-party services.
  - Wrote automated tests using Jest and Pytest.
  - Used PostgreSQL, Redis, Docker, and 

## 7. Chain-of-thought prompting

Chain-of-thought (CoT) prompting asks the model to break a task into intermediate steps before producing an answer. The technique is most useful when the task needs several linked operations—for example, identify each role, preserve its dates, separate tools from responsibilities, and distinguish professional use from personal experimentation.

A case for using it: a one-line request such as “summarize Alex's experience and stack” leaves several decisions implicit. A stepwise prompt makes the desired process explicit, which can improve completeness on a multi-part task. On a simple lookup, the extra steps may add little.

For a teaching example, the prompt below names the steps. The response asks for a concise audit summary with evidence, rather than a long private reasoning transcript.

In [8]:
cot_prompt = f"""Analyze the profile using this checklist before writing the result:
1. Find each job title, employer, date range, and main responsibilities.
2. List every named technology and group it by category.
3. Separate tools used at work from tools mentioned only in personal projects.
4. Check that each item in the answer is supported by the profile.

Return only:
- Work Experience: concise bullets for each role.
- Technology Stack: categorized bullets, marking personal-project-only tools.
- Evidence Check: one brief note about how unsupported claims were avoided.

Profile:
{profile}"""

print(ask(cot_prompt))

### Work Experience
- **Senior Software Engineer at Northstar Health** (March 2022–present)
  - Builds and maintains patient-facing web features and internal operational tools.
  - Develops backend APIs using Python (FastAPI, Django).
  - Utilizes PostgreSQL for data management and Redis for caching/job queues.
  - Deploys applications to AWS with Docker and Kubernetes.
  - Implements CI/CD using GitHub Actions and manages infrastructure with Terraform.
  - Reviews code and mentors junior engineers.

- **Software Engineer at BrightMarket** (July 2019–February 2022)
  - Developed customer and merchant features for an online marketplace.
  - Implemented web interfaces using TypeScript and React, and APIs with Node.js (Express).
  - Integrated REST endpoints with third-party payment and shipping services.
  - Wrote automated tests using Jest and Pytest.
  - Worked in Agile sprints and utilized PostgreSQL, Redis, Docker, and AWS.

- **Junior Web Developer at Cedar Labs** (June 2017–June 20

### What this CoT example demonstrates

The prompt decomposes one broad extraction task into smaller checks, then defines a compact answer format. Compare its completeness and organization with the zero-shot result. The prompt can encourage a more systematic answer, but the output still needs checking against the source.

## 8. Contextual or grounded prompting

A contextual prompt supplies source material and tells the model to answer from that material. The profile itself is the context. Grounding instructions are useful for extraction because they set a boundary around the evidence the model should use.

In [9]:
grounded_prompt = f"""Using only the profile below, extract Alex's work experience and
technology stack. Do not infer skill level or add common tools that are not named.
If a technology is mentioned only for personal projects, label it that way.

Profile:
{profile}"""

print(ask(grounded_prompt))

### Work Experience

**Current Work**  
**Senior Software Engineer at Northstar Health**  
March 2022–present  
- Builds and maintains patient-facing web features and internal operational tools.  
- Develops backend APIs in Python using FastAPI and Django.  
- Works with PostgreSQL and Redis.  
- Deploys services to AWS using Docker and Kubernetes.  
- Uses GitHub Actions for continuous integration and Terraform.  

**Previous Work**  
**Software Engineer at BrightMarket**  
July 2019–February 2022  
- Built customer and merchant features for an online marketplace.  
- Used TypeScript and React for web interfaces, and Node.js with Express for APIs.  
- Integrated REST endpoints with third-party payment and shipping providers.  
- Used PostgreSQL, Redis, Docker, and AWS.  

**Earlier Experience**  
**Junior Web Developer at Cedar Labs**  
June 2017–June 2019  
- Created responsive websites using JavaScript, HTML, CSS, and Bootstrap.  
- Maintained a PHP application and MySQL database.  

## 9. Output-format prompting

A prompt can request a specific format, such as JSON. This is useful when the result will be consumed by another program. A natural-language request alone does not guarantee valid JSON; production code should use structured outputs and validate the result.

In [10]:
import json

json_prompt = f"""Extract Alex's experience and technology stack from the profile.
Return only a JSON object with this structure:
{{
  "work_experience": [
    {{
      "title": "string",
      "employer": "string",
      "dates": "string",
      "responsibilities": ["string"]
    }}
  ],
  "technology_stack": {{
    "category": ["technology"]
  }},
  "personal_project_only": ["technology"]
}}
Use only profile facts. Do not include Markdown fences.

Profile:
{profile}"""

json_text = ask(json_prompt)
print(json_text)

try:
    result = json.loads(json_text)
    print("\nValid JSON parsed successfully.")
except json.JSONDecodeError as error:
    print("\nThe response was not valid JSON:", error)

{
  "work_experience": [
    {
      "title": "Senior Software Engineer",
      "employer": "Northstar Health",
      "dates": "March 2022–present",
      "responsibilities": [
        "Builds and maintains patient-facing web features and internal operational tools.",
        "Develops backend APIs in Python using FastAPI and Django.",
        "Works with PostgreSQL for transactional data and Redis for caching and job queues.",
        "Deploys services to AWS using Docker and Kubernetes.",
        "Uses GitHub Actions for continuous integration and Terraform to manage cloud infrastructure.",
        "Collaborates with product managers, designers, QA, and security engineers.",
        "Reviews code and mentors two junior engineers.",
        "Helped split a large Django application into smaller services."
      ]
    },
    {
      "title": "Software Engineer",
      "employer": "BrightMarket",
      "dates": "July 2019–February 2022",
      "responsibilities": [
        "Built custome

## 10. Compare the techniques

| Technique | What changes in the prompt | What to observe |
| --- | --- | --- |
| Zero-shot | Task only | What format and detail does the model choose by itself? |
| Few-shot | Task plus examples | Does it follow the demonstrated structure? |
| Role/persona | Adds a perspective | Does focus or tone change while facts stay grounded? |
| Constraints | Adds explicit rules | Are unsupported or out-of-scope details reduced? |
| Chain of thought | Breaks the task into steps | Is a multi-part result more complete and organized? |
| Contextual/grounded | Emphasizes the supplied source | Does the answer stay within the profile? |
| Output format | Specifies JSON fields | Is the result easier to parse, and does it parse successfully? |

These patterns can be combined. For example, a few-shot prompt can also include constraints and request a specific output format.

## Try it yourself

1. Add a new job to the profile and compare all prompt styles.
2. Add a rule that the model must preserve date ranges exactly as written.
3. Ask for a short professional summary, then compare it with the extracted facts.
4. Remove the examples from the few-shot prompt. What changed?
5. Modify the CoT checklist to make the evidence check more explicit.

Model output can vary between runs. Check important facts against the profile, especially dates, tool names, and whether experience was professional or personal.

## Troubleshooting

- If `OpenAI()` reports a missing API key, set `OPENAI_API_KEY` in the environment or in a local `.env` file.
- If imports fail, install `openai` and `python-dotenv` in the active notebook environment.
- Each cell containing `ask(...)` sends a model request and may incur charges.
- The profile is fictional and is included only as a teaching example.

## 11. Combine the techniques into one prompt

A strong prompt can combine a role, a clear action, source material, rules for uncertainty, a response structure, and a quality check. This capstone asks for exactly three sections: overall experience, a skill and domain analyzer, and a profile summary/classification. It also asks the model to score its result inside the Skill Analyzer section. Treat that score as a review aid, not an independently verified measure of correctness.

In [13]:
from datetime import date

as_of_date = date.today().isoformat()

combined_prompt = f"""
ROLE
You are a careful technical recruiter and skills analyst. Produce a concise,
evidence-based summary of the person in the source.

ACTION
Calculate overall professional experience and approximate experience for
each explicitly named skill. Identify relevant domain exposure. Then write
a short profile gist and classify the strongest supported professional
orientation (for example: Coder, Product Manager, Leader, or Architect).

SOURCE
As-of date for experience estimates: {as_of_date}
<profile>
{profile}
</profile>

METHOD
1. Calculate each dated role duration through the as-of date for a current role,
   or through its stated end month for a completed role. Dates are month-only,
   so report approximate years rounded to one decimal place.
2. Add non-overlapping role durations for total experience. Cross-check the
   total against any summary claim and explicitly report discrepancies.
3. For each skill, add the durations of roles where the profile explicitly
   states it was used; never invent a shorter duration or count overlapping
   calendar periods twice. Use the full role duration as an estimate only.
4. Treat date-derived years as approximate exposure, not proof of continuous
   use or proficiency. Keep personal projects separate.
5. Assign a time-based maturity label: Beginner = under 1 year; Intermediate =
   1 to under 3 years; Seasoned = 3 or more years. For missing/undated evidence,
   use Not enough information rather than guessing.
6. Before responding, check dates, calculations, every named skill/domain, and
   all three required sections against the source.

RULES FOR AMBIGUITY
- Use only facts in the source; do not infer tools, proficiency, or job duties.
- If the profile's summary conflicts with its dated work history, calculate
  from the dated roles and flag the discrepancy.
- If dates are incomplete, report an approximate range and say why; do not
  imply precision the source does not support.
- If skill duration cannot be tied to dated roles, write Not enough information.
- Separate professional experience from personal-project-only exposure.
- Describe domain knowledge as role-based exposure, not certified expertise.
- For the profile classification, choose only labels supported by evidence;
  include a brief reason and say Not evidenced for unsupported categories.

SCORING RUBRIC (include this within section 2)
Score the response out of 100: accuracy 0-40, completeness 0-25, evidence
quality 0-20, and format compliance 0-15. Give a short reason per score and
the total. This is a self-assessment, not independent verification.

RESPONSE STRUCTURE — RETURN EXACTLY THESE THREE TOP-LEVEL SECTIONS
1. Experience
   - Total professional experience: years (and months if useful).
   - Overall maturity: Beginner, Intermediate, or Seasoned, using the stated
     time bands; explain briefly that this is a time-based proxy.
   - Mention any conflict between the stated summary and dated roles.
2. Skill Analyzer
   - Table columns: Skill/Domain | Approx. years | Evidence or role(s) | Maturity.
   - Include every named technology, tools, frameworks, and relevant domains.
   - Mark personal-project-only skills and undated skills clearly.
   - End this section with Quality Score: rubric scores, reasons, and total.
3. Profile
   - Give a 2-3 sentence gist of what the person does well.
   - Name the best-supported orientation (Coder, Product Manager, Leader,
     Architect, or another evidence-based label) and cite brief evidence.

Do not add other top-level sections. Keep all claims concise and factual.
"""

print(ask(combined_prompt))

### 1. Experience
- **Total professional experience:** 7 years and 4 months.
- **Overall maturity:** Seasoned. This classification is based on over three years of cumulative experience in relevant software engineering roles.
- There is no conflict between the stated summary and dated roles.

### 2. Skill Analyzer
| Skill/Domain          | Approx. years | Evidence or role(s)                         | Maturity   |
|----------------------|---------------|---------------------------------------------|------------|
| Web Applications      | 7.4           | Northstar Health, BrightMarket, Cedar Labs | Seasoned   |
| Data Services         | 7.4           | Northstar Health, BrightMarket              | Seasoned   |
| Python                | 4.7           | Northstar Health                            | Seasoned   |
| FastAPI              | 1.6           | Northstar Health                            | Intermediate|
| Django                | 3.5           | Northstar Health, BrightMarket         